# 07b — Sentinel-5P CDSE Window Extract

April 2025 replacement.

In [ ]:

SITE_ID = "NHV"
SITE_NAME = "Newhaven ERF"
LAT = 50.80208
LON = 0.04961
PRODUCT_KEY = "NO2"
TIMELINESS = "OFFL"
DATE_FROM = "2025-04-02"
DATE_TO = "2025-04-07"
BUFFER_DEG = 0.20
MAX_SCENES = 100
DOWNLOAD_FIRST_PRODUCT = False
OUTPUT_DIR = "outputs/s5p"


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import os, json
import requests
import pandas as pd
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:

CDSE_USERNAME = os.getenv("CDSE_USERNAME") or os.getenv("CDSE_USER")
CDSE_PASSWORD = os.getenv("CDSE_PASSWORD")
assert CDSE_USERNAME and CDSE_PASSWORD
TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
ODATA_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
r = requests.post(TOKEN_URL, data={"client_id":"cdse-public","username":CDSE_USERNAME,"password":CDSE_PASSWORD,"grant_type":"password"}, timeout=60); r.raise_for_status(); token=r.json()["access_token"]
PRODUCT_PATTERNS = {("NO2","OFFL"):"L2__NO2___",("NO2","NRTI"):"L2__NO2___",("SO2","OFFL"):"L2__SO2___",("SO2","NRTI"):"L2__SO2___",("CO","OFFL"):"L2__CO____",("CO","NRTI"):"L2__CO____"}
name_pat = PRODUCT_PATTERNS[(PRODUCT_KEY.upper(), TIMELINESS.upper())]
minx,maxx=LON-BUFFER_DEG,LON+BUFFER_DEG; miny,maxy=LAT-BUFFER_DEG,LAT+BUFFER_DEG
wkt=f"POLYGON(({minx} {miny},{maxx} {miny},{maxx} {maxy},{minx} {maxy},{minx} {miny}))"
flt=("Collection/Name eq 'SENTINEL-5P' " + f"and ContentDate/Start ge {DATE_FROM}T00:00:00.000Z " + f"and ContentDate/Start le {DATE_TO}T23:59:59.999Z " + f"and OData.CSC.Intersects(area=geography'SRID=4326;{wkt}') " + f"and contains(Name,'{name_pat}') " + f"and contains(Name,'_{TIMELINESS.upper()}_')")
r = requests.get(ODATA_URL, params={"$filter":flt,"$top":str(MAX_SCENES),"$orderby":"ContentDate/Start asc","$count":"true"}, headers={"Authorization":f"Bearer {token}"}, timeout=120); r.raise_for_status()
value=r.json().get("value",[])
rows=[{"site_id":SITE_ID,"site_name":SITE_NAME,"lat":LAT,"lon":LON,"product_key":PRODUCT_KEY.upper(),"timeliness":TIMELINESS.upper(),"product_id":p.get("Id"),"name":p.get("Name"),"s3_path":p.get("S3Path"),"start_utc":p.get("ContentDate",{}).get("Start"),"end_utc":p.get("ContentDate",{}).get("End"),"published_utc":p.get("PublicationDate"),"online":p.get("Online"),"size_bytes":p.get("ContentLength"),"footprint_wkt":p.get("GeoFootprint")} for p in value]
df=pd.DataFrame(rows)
csv_path=Path(OUTPUT_DIR)/f"{SITE_ID}_{PRODUCT_KEY.lower()}_{TIMELINESS.lower()}_scene_catalog.csv"
pq_path=Path(OUTPUT_DIR)/f"{SITE_ID}_{PRODUCT_KEY.lower()}_{TIMELINESS.lower()}_scene_catalog.parquet"
js_path=Path(OUTPUT_DIR)/f"{SITE_ID}_{PRODUCT_KEY.lower()}_{TIMELINESS.lower()}_scene_catalog.json"
df.to_csv(csv_path,index=False); df.to_parquet(pq_path,index=False); js_path.write_text(json.dumps(value,indent=2),encoding="utf-8")
(Path(OUTPUT_DIR)/f"{SITE_ID}_{PRODUCT_KEY.lower()}_{TIMELINESS.lower()}_manifest.json").write_text(json.dumps({"created_utc":datetime.now(timezone.utc).isoformat(),"site_id":SITE_ID,"site_name":SITE_NAME,"lat":LAT,"lon":LON,"date_from":DATE_FROM,"date_to":DATE_TO,"buffer_deg":BUFFER_DEG,"product_key":PRODUCT_KEY.upper(),"timeliness":TIMELINESS.upper(),"scene_count":int(len(df))},indent=2),encoding="utf-8")
display(df.head(20))
